# Sales & Demand Forecasting for a UK Online Retailer

**Prepared for:** store owners, startup founders, and business managers who need a clear, no-jargon view of where sales are headed and how to plan around it.

**What this notebook does:** cleans a year of real transaction data, engineers time-based features (trend, seasonality, day-of-week effects), trains and evaluates forecasting models, and produces a 30-day forecast with business-facing charts and recommendations.

**Dataset:** [UCI Online Retail](https://archive.ics.uci.edu/ml/datasets/online+retail) — ~542,000 line-item transactions from a UK-based online retailer, December 2010 to December 2011.


In [1]:
import warnings
warnings.filterwarnings("ignore")

import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from viz_style import apply_style, currency_axis, SERIES_1, SERIES_2, SERIES_2_BAND

apply_style()
pd.set_option("display.max_columns", 20)


## 1. Load the raw data

In [2]:
raw = pd.read_excel("../data/raw/online_retail.xlsx")
print(f"Rows: {len(raw):,}  |  Columns: {list(raw.columns)}")
print(f"Date range: {raw['InvoiceDate'].min()} to {raw['InvoiceDate'].max()}")
raw.head()


Rows: 541,909  |  Columns: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']
Date range: 2010-12-01 08:26:00 to 2011-12-09 12:50:00


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [3]:
raw.info()
print()
raw.describe(include="all").T


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB



,count,unique,top,freq,mean,min,25%,50%,75%,max,std
InvoiceNo,541909.0,25900.0,573585.0,1114.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
StockCode,541909,4070,85123A,2313,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Description,540455,4223,WHITE HANGING HEART T-LIGHT HOLDER,2369,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Quantity,541909.0,NaN,NaN,NaN,9.55225,-80995.0,1.0,3.0,10.0,80995.0,218.081158
InvoiceDate,541909,NaN,NaN,NaN,2011-07-04 13:34:57.156386048,2010-12-01 08:26:00,2011-03-28 11:34:00,2011-07-19 17:17:00,2011-10-19 11:27:00,2011-12-09 12:50:00,NaN
UnitPrice,541909.0,NaN,NaN,NaN,4.611114,-11062.06,1.25,2.08,4.13,38970.0,96.759853
CustomerID,406829.0,NaN,NaN,NaN,15287.69057,12346.0,13953.0,15152.0,16791.0,18287.0,1713.600303
Country,541909,38,United Kingdom,495478,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Clean the transaction data

Raw retail transaction exports always need a pass of cleaning before they can be trusted:

- **Cancellations** — `InvoiceNo` values starting with `"C"` are returns/cancellations, not sales.
- **Bad quantities/prices** — a handful of rows have `Quantity <= 0` or `UnitPrice <= 0` (adjustments, data-entry errors); these aren't real sales either.
- **Missing product descriptions** — a small number of rows have no `Description`, usually bookkeeping adjustments rather than product sales.

`CustomerID` is missing for many rows, but that's irrelevant here — we're forecasting total revenue, not building a customer-level model, so we keep those rows.


In [4]:
n_before = len(raw)

df = raw.copy()
df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]
df = df[df["Description"].notna()]
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

n_after = len(df)
pct_dropped = (n_before - n_after) / n_before * 100
print(f"Rows before cleaning: {n_before:,}")
print(f"Rows after cleaning:  {n_after:,}")
print(f"Dropped: {n_before - n_after:,} rows ({pct_dropped:.1f}%)")

assert (df["Revenue"] < 0).sum() == 0, "Negative revenue remains after cleaning"
assert df["Description"].isna().sum() == 0, "Missing descriptions remain after cleaning"
print("Checks passed: no negative revenue, no missing descriptions.")


Rows before cleaning: 541,909
Rows after cleaning:  530,104
Dropped: 11,805 rows (2.2%)
Checks passed: no negative revenue, no missing descriptions.


## 3. Build the daily revenue series

The task is to forecast **UK daily revenue** (about 91% of all transactions), since a single clear market tells a cleaner story than blending currencies and geographies. We also pull two descriptive-only summaries (top countries, top products) to support the business narrative later — those are not forecast individually.

The transaction data is at the line-item level, so we aggregate to one revenue total per calendar day, then fill in any day with zero recorded transactions as `0` (rather than silently dropping it) so the series has a regular daily frequency — required for lag/rolling features later. The final month (December 2011) is cut off mid-month in the source data, so we truncate the series at the last **complete** month to avoid the model mistaking a data cutoff for a demand collapse.


In [5]:
uk = df[df["Country"] == "United Kingdom"].copy()
uk["Date"] = uk["InvoiceDate"].dt.normalize()

daily = uk.groupby("Date")["Revenue"].sum()

full_range = pd.date_range(daily.index.min(), daily.index.max(), freq="D")
daily = daily.reindex(full_range, fill_value=0.0)
daily.index.name = "Date"

# Truncate the trailing partial month (Dec 2011 only has data through the 9th).
last_full_month_end = (daily.index.max().replace(day=1) - pd.Timedelta(days=1))
daily = daily.loc[:last_full_month_end]

assert len(daily) == (daily.index.max() - daily.index.min()).days + 1, "Gaps in the daily date index"
assert daily.isna().sum() == 0, "Unexpected NaNs in daily revenue series"

print(f"Daily series: {daily.index.min().date()} to {daily.index.max().date()}  ({len(daily)} days)")
print(f"Total revenue: £{daily.sum():,.0f}  |  Average day: £{daily.mean():,.0f}")

daily_revenue = daily.rename("Revenue").reset_index()
daily_revenue.to_csv("../data/processed/daily_revenue.csv", index=False)
daily_revenue.tail()


Daily series: 2010-12-01 to 2011-11-30  (365 days)
Total revenue: £8,432,600  |  Average day: £23,103


,Date,Revenue
360,2011-11-26,0.00
361,2011-11-27,19781.17
362,2011-11-28,52649.14
363,2011-11-29,65304.01
364,2011-11-30,50546.43


In [6]:
top_countries = (
    df.groupby("Country")["Revenue"].sum().sort_values(ascending=False).head(5)
)
top_products_uk = (
    uk.groupby("Description")["Revenue"].sum().sort_values(ascending=False).head(10)
)

print("Top 5 countries by revenue:")
print(top_countries.apply(lambda x: f"£{x:,.0f}"))
print()
print("Top 10 products by revenue (UK):")
print(top_products_uk.apply(lambda x: f"£{x:,.0f}"))


Top 5 countries by revenue:
Country
United Kingdom    £9,025,222
Netherlands         £285,446
EIRE                £283,454
Germany             £228,867
France              £209,715
Name: Revenue, dtype: object

Top 10 products by revenue (UK):
Description
DOTCOM POSTAGE                        £206,249
PAPER CRAFT , LITTLE BIRDIE           £168,470
REGENCY CAKESTAND 3 TIER              £142,273
WHITE HANGING HEART T-LIGHT HOLDER    £100,498
PARTY BUNTING                          £93,659
JUMBO BAG RED RETROSPOT                £86,471
MEDIUM CERAMIC TOP STORAGE JAR         £80,576
PAPER CHAIN KIT 50'S CHRISTMAS         £62,743
ASSORTED COLOUR BIRD ORNAMENT          £54,757
CHILLI LIGHTS                          £53,337
Name: Revenue, dtype: object
